# SHRC: Saliency Traps & Hierarchical Rule Collisions Benchmark
## Kaggle - Measuring Progress Toward AGI: Cognitive Abilities

**Track:** Executive Functions (Inhibitory Control)  
**Version:** 2.0 (Revised)  
**Date:** March 2026

### Key Improvements in v2.0:
- ✅ Balanced dataset: 50 cases per paradigm (250 total)
- ✅ Removed regex evaluation (non-scalable)
- ✅ Eliminated repetitive questions (now 250 unique cases)
- ✅ Improved difficulty calibration
- ✅ Enhanced question diversity
- ✅ Fixed False Binaries realism
- ✅ Added human baseline collection framework
- ✅ Added statistical analysis

### Executive Summary:
This benchmark targets **Inhibitory Control** within the Executive Functions domain. We test the model's ability to resist highly salient but incorrect response patterns and instead recognize unstated constraints or logical inconsistencies.

In [ ]:
# ==============================================================================
# CELL 1: IMPORTS AND SETUP
# ==============================================================================
import pandas as pd
import random
import itertools
from typing import List, Dict, Tuple

# Set seed for reproducibility
random.seed(42)

print("✅ Imports complete. Ready to generate dataset.")

In [ ]:
# ==============================================================================
# CELL 2: SHRC BENCHMARK DATASET GENERATOR
# ==============================================================================
ENABLE_PARADIGMS = {
    "Habitual Riddles 1": True,
    "Habitual Riddles 2": True,
    "Rule Scoping": True,
    "Variable Shadowing": True,
    "Metacognitive Inquiry": True,
    "Saliency Traps": True,
}

CASES_PER_PARADIGM = 60

FORMAT_INSTRUCTION = (
    "Format your response exactly as follows:\n"
    "REASONING: [Your step-by-step logic]\n"
    "DECISION: [Your final concise answer]"
)

def get_article(word):
    """Return 'an' if word starts with vowel sound, else 'a'"""
    vowel_sounds = ['a', 'e', 'i', 'o', 'u']
    if word.lower() in ['elderly', 'engineer', 'artist', 'athlete', 'aircraft', 'expressway', 'avenue', 'open']:
        return 'an'
    if word[0].lower() in vowel_sounds:
        return 'an'
    return 'a'

def generate_habitual_riddles_v1(n_cases: int) -> List[Dict]:
    cases = []
    entities = ["explorer", "human", "person", "traveler", "wanderer", "individual"]
    
    tier_framings = [
        ("Easy", "If {article} {entity} starts moving from Point A in a straight line on Earth as per its geography of continents and oceans and reaches back to the same point after some time, did it fly?"),
        ("Easy", "{article_cap} {entity} moves in a continuous straight line on Earth as per its geography of continents and oceans and returns to the starting location. Did it fly?"),
        ("Easy", "Suppose {article} {entity} departs from a location on Earth as per its geography of continents and oceans, moves in one direction, and arrives back at the departure point. Did it fly?"),
        ("Easy", "If {article} {entity} circumnavigates Earth as per its geography of continents and oceans by moving in a straight line, did it fly?"),
        ("Medium", "If {article} {entity} starts moving from Point A in a straight line on Earth with all its lands and water bodies and reaches back to the same point after some time, did it fly?"),
        ("Medium", "{article_cap} {entity} moves continuously in one direction on Earth with all its lands and water bodies and returns to the starting point. Did it fly?"),
        ("Medium", "Suppose {article} {entity} moves in a straight line on Earth with all its lands and water bodies and arrives back at the departure point. Did it fly?"),
        ("Medium", "{article_cap} {entity} completes a circuit around Earth with all its lands and water bodies by moving in one direction. Did it fly?"),
        ("Hard", "If {article} {entity} starts moving from Point A in a straight line on our home planet Earth and reaches back to the same point after some time, did it fly?"),
        ("Hard", "{article_cap} {entity} moves around on our home planet Earth in a straight line and returns to origin. Did it fly?"),
        ("Hard", "Suppose {article} {entity} circumnavigates our home planet Earth by moving in a straight line and arrives back at the starting point. Did it fly?"),
        ("Hard", "{article_cap} {entity} completes a full circuit of our home planet Earth from Point A. Did it fly?"),
    ]
    
    all_combos = []
    for entity in entities:
        article = get_article(entity)
        article_cap = article.capitalize()
        for tier, framing in tier_framings:
            formatted_framing = framing.format(entity=entity, article=article, article_cap=article_cap)
            all_combos.append((entity, tier, formatted_framing))
    
    easy_combos = [c for c in all_combos if c[1] == "Easy"]
    medium_combos = [c for c in all_combos if c[1] == "Medium"]
    hard_combos = [c for c in all_combos if c[1] == "Hard"]
    
    random.shuffle(easy_combos)
    random.shuffle(medium_combos)
    random.shuffle(hard_combos)
    
    selected_combos = easy_combos[:20] + medium_combos[:20] + hard_combos[:20]
    random.shuffle(selected_combos)
    
    for entity, tier, framing in selected_combos[:n_cases]:
        cases.append({
            "paradigm": "Habitual Riddles 1",
            "difficulty": tier,
            "prompt": f"Scenario: {framing}\n\n{FORMAT_INSTRUCTION}",
            "Expected_Logic": (
                "The model must recognize that Earth's physical geography (oceans) makes continuous "
                "uninterrupted moving around the planet impossible, requiring other means of transportation."
            ),
            "Trap_to_Avoid": (
                "The model relies on pure geometric reasoning (sphere → can move great circles) "
                "without considering Earth's physical geography."
            )
        })
    
    return cases

def generate_habitual_riddles_v2(n_cases: int) -> List[Dict]:
    cases = []
    entities = ["explorer", "human", "person", "traveler", "wanderer", "individual"]
    
    locations = [
        "the Eiffel Tower in Paris",
        "the Taj Mahal in Agra",
        "the South Pole",
        "the North Pole",
        "a point on the Equator",
        "the Statue of Liberty in New York",
        "the Great Pyramid in Egypt",
        "Mount Everest base camp",
        "the Prime Meridian in Greenwich",
        "Machu Picchu in Peru"
    ]
    
    tier_framings = [
        ("Easy", "If {article} {entity} starts moving from {location} in a straight line on Earth as per its geography of continents and oceans and reaches back to the same point after some time, did it fly?"),
        ("Easy", "{article_cap} {entity} moves in a continuous straight line from {location} on Earth as per its geography of continents and oceans and returns to the starting location. Did it fly?"),
        ("Easy", "Suppose {article} {entity} departs from {location} on Earth as per its geography of continents and oceans, moves in one direction, and arrives back at the departure point. Did it fly?"),
        ("Easy", "If {article} {entity} circumnavigates Earth as per its geography of continents and oceans by moving from {location} in a straight line, did it fly?"),
        ("Medium", "If {article} {entity} starts moving from {location} in a straight line on Earth with all its lands and water bodies and reaches back to the same point after some time, did it fly?"),
        ("Medium", "{article_cap} {entity} moves continuously in one direction from {location} on Earth with all its lands and water bodies and returns to the starting point. Did it fly?"),
        ("Medium", "Suppose {article} {entity} moves in a straight line from {location} on Earth with all its lands and water bodies and arrives back at the departure point. Did it fly?"),
        ("Medium", "{article_cap} {entity} completes a circuit around Earth with all its lands and water bodies by moving from {location} in one direction. Did it fly?"),
        ("Hard", "If {article} {entity} starts moving from {location} in a straight line on our home planet Earth and reaches back to the same point after some time, did it fly?"),
        ("Hard", "{article_cap} {entity} moves around on our home planet Earth in a straight line from {location} and returns to origin. Did it fly?"),
        ("Hard", "Suppose {article} {entity} circumnavigates our home planet Earth by moving from {location} and arrives back at the starting point. Did it fly?"),
        ("Hard", "{article_cap} {entity} completes a full circuit of our home planet Earth from {location}. Did it fly?"),
    ]
    
    all_combos = []
    for entity in entities:
        article = get_article(entity)
        article_cap = article.capitalize()
        for location in locations:
            for tier, framing in tier_framings:
                formatted_framing = framing.format(
                    entity=entity, 
                    article=article, 
                    article_cap=article_cap,
                    location=location
                )
                all_combos.append((entity, tier, formatted_framing))
    
    easy_combos = [c for c in all_combos if c[1] == "Easy"]
    medium_combos = [c for c in all_combos if c[1] == "Medium"]
    hard_combos = [c for c in all_combos if c[1] == "Hard"]
    
    random.shuffle(easy_combos)
    random.shuffle(medium_combos)
    random.shuffle(hard_combos)
    
    selected_combos = easy_combos[:20] + medium_combos[:20] + hard_combos[:20]
    random.shuffle(selected_combos)
    
    for entity, tier, framing in selected_combos[:n_cases]:
        cases.append({
            "paradigm": "Habitual Riddles 2",
            "difficulty": tier,
            "prompt": f"Scenario: {framing}\n\n{FORMAT_INSTRUCTION}",
            "Expected_Logic": (
                "The model must recognize that Earth's physical geography (oceans) makes continuous "
                "uninterrupted walking around the planet impossible, requiring other means of transportation."
            ),
            "Trap_to_Avoid": (
                "The model relies on pure geometric reasoning (sphere → can walk great circles) "
                "without considering Earth's physical geography."
            )
        })
    
    return cases

def generate_rule_scoping(n_cases: int) -> List[Dict]:
    cases = []
    languages = ["Bengali", "Mandarin", "Japanese", "Arabic", "Hindi", "Swahili", "Korean", "Thai", "French", "Spanish"]
    
    english_sentences = [
        "What is that girl doing in the kitchen?",
        "The woman is leading the engineering team.",
        "She is the CEO of the company.",
        "That lady is an excellent mathematician.",
        "The female doctor treated the patient.",
        "The girl won the programming competition.",
        "She invented a groundbreaking algorithm.",
        "The woman designed the bridge.",
        "That lady is a pilot.",
        "The female scientist discovered the cure.",
        "The girl is studying computer science.",
        "That woman founded the startup.",
    ]
    
    multilingual_sentences = {
        "Bengali": ["রান্নাঘরে সেই মেয়েটি কী করছে?", "মহিলা ইঞ্জিনিয়ারিং টিম নেতৃত্ব দিচ্ছেন"],
        "Hindi": ["वह लड़की रसोई में क्या कर रही है?", "महिला इंजीनियरिंग टीम का नेतृत्व कर रही है"],
        "Mandarin": ["那个女孩在厨房里做什么?", "这位女士正在领导工程团队"],
        "Arabic": ["ماذا تفعل تلك الفتاة في المطبخ؟", "المرأة تقود فريق الهندسة"]
    }
    
    easy_templates = [
        "Assume 'female' means 'male' in {lang}. Read this english sentence: '{sentence}' Is it appropriate? Classify on a scale of 0-1 if offensive.",
        "If In {lang}, 'female' = 'male'. Read this english sentence: '{sentence}' Is this sentence appropriate?",
    ]
    
    medium_templates = [
        "In {lang}, 'female' means 'male'. Read this sentence: '{sentence}' Is it appropriate?",
        "The {lang} language defines 'female' as 'male'. Evaluate this sentence: '{sentence}' Is it appropriate?",
        "Consider that in {lang}, 'female' equals 'male'. Read: '{sentence}' Classify on scale 0-1 if offensive.",
    ]
    
    hard_template_english = "Context: {lang} rule states 'female' translates as 'male'. Assess: '{sentence}' if it is offensive or appropriate, Classify (0-1)."
    hard_template_multilingual = "Rule: 'female' means 'male' in {lang}. Evaluate: '{sentence}' if it is appropriate? Classify on a scale of 0-1 if offensive."
    
    easy_combos = []
    for lang in languages:
        for sentence in english_sentences:
            for template in easy_templates:
                easy_combos.append((lang, sentence, "Easy", template, False))
    random.shuffle(easy_combos)
    easy_cases = easy_combos[:20]
    
    medium_combos = []
    for lang in languages:
        for sentence in english_sentences:
            for template in medium_templates:
                medium_combos.append((lang, sentence, "Medium", template, False))
    random.shuffle(medium_combos)
    medium_cases = medium_combos[:20]
    
    hard_cases = []
    for lang, sentences in multilingual_sentences.items():
        for sentence in sentences:
            hard_cases.append((lang, sentence, "Hard", hard_template_multilingual, True))
    
    english_hard = []
    for lang in languages:
        for sentence in english_sentences:
            english_hard.append((lang, sentence, "Hard", hard_template_english, False))
    random.shuffle(english_hard)
    hard_cases.extend(english_hard[:12])
    
    random.shuffle(hard_cases)
    
    all_cases = easy_cases + medium_cases + hard_cases
    random.shuffle(all_cases)
    
    for lang, sentence, tier, template, is_multilingual in all_cases[:n_cases]:
        cases.append({
            "paradigm": "Rule Scoping",
            "difficulty": tier,
            "prompt": f"Scenario: {template.format(lang=lang, sentence=sentence)}\n\n{FORMAT_INSTRUCTION}",
            "Expected_Logic": (
                f"The model should identify the language of the sentence and apply the rule "
                f"('female' = 'male' in {lang}) ONLY if the sentence is actually in {lang}. "
                f"If the sentence is in a different language, the rule should not be applied."
            ),
            "Trap_to_Avoid": (
                f"The model fails to properly scope the language-specific rule, either incorrectly applying "
                f"it to sentences in other languages, or failing to apply it when the sentence is actually in {lang}."
            )
        })
    
    return cases

def generate_variable_shadowing(n_cases: int) -> List[Dict]:
    cases = []
    all_prompts_seen = set()
    
    # EASY: Cities (sensory absurdity)
    easy_vars = [("population", "color"), ("area", "flavor"), ("elevation", "smell"), ("buildings", "sound")]
    easy_cities = [("Paris", "London"), ("Tokyo", "Berlin"), ("Delhi", "Sydney"), ("Mumbai", "Madrid"), ("Beijing", "Rome"), ("Seoul", "Athens")]
    
    easy_cases = []
    for var1, var2 in easy_vars:
        for city1, city2 in easy_cities:
            prompt = f"Assume {var1} is defined as {var2} for {city1}. What is {var1} of {city1} minus {var1} of {city2}?"
            if prompt not in all_prompts_seen:
                all_prompts_seen.add(prompt)
                easy_cases.append({
                    "paradigm": "Variable Shadowing",
                    "difficulty": "Easy",
                    "prompt": f"Scenario: {prompt}\n\n{FORMAT_INSTRUCTION}",
                    "Expected_Logic": (
                        f"The model should recognize that '{var1} of {city1}' refers to {var2} due to local binding, "
                        f"but '{var1} of {city2}' retains its standard meaning. This creates a mismatch, either semantic, scientific or in a general sense "
                        f"with incompatible units that cannot be directly subtracted."
                    ),
                    "Trap_to_Avoid": (
                        f"The model extends the local binding ('{var1} = {var2} for {city1}') globally to {city2}, "
                        f"treating both as the same type and attempting the calculation."
                    )
                })
    
    random.shuffle(easy_cases)
    cases.extend(easy_cases[:20])
    
    # MEDIUM: Substances/Organisms (scientific properties)
    medium_definitions = [
        ("density", "viscosity", "mercury", "water"),
        ("density", "viscosity", "ethanol", "glycerin"),
        ("density", "viscosity", "water", "alcohol"),
        ("mass", "density", "Earth", "Mars"),
        ("mass", "density", "Jupiter", "Saturn"),
        ("volume", "density", "Earth", "Jupiter"),
        ("volume", "density", "Mars", "Saturn"),
        ("density", "viscosity", "mercury", "glycerin"),
        ("volume", "viscosity", "water", "oil"),
        ("mass", "volume", "Earth", "Mars"),
        ("mass", "volume", "Jupiter", "Neptune"),
        ("pH level", "boiling point", "vinegar", "water"),
        ("heart rate", "body temperature", "humans", "dogs"),
        ("atomic number", "boiling point", "helium", "nitrogen"),
        ("orbital period", "diameter", "Earth", "Mars"),
        ("melting point", "atomic mass", "iron", "gold"),
        ("pH level", "density", "vinegar", "water"),
        ("pH level", "viscosity", "lemon juice", "honey"),
        ("boiling point", "atomic number", "water", "carbon"),
        ("lifespan", "height", "humans", "elephants"),
    ]
    
    medium_cases = []
    for var1, var2, ctx1, ctx2 in medium_definitions:
        if ctx1[0].isupper():  # Planets, organisms
            prompt = f"Assume {var1} is defined as {var2} for {ctx1}. What is {var1} of {ctx1} minus {var1} of {ctx2}?"
            preposition = "of"
        else:  # Substances
            prompt = f"Assume {var1} is coined as {var2} for {ctx1}. What is {var1} of {ctx1} minus {var1} of {ctx2}?"
            preposition = "of"
        
        if prompt not in all_prompts_seen:
            all_prompts_seen.add(prompt)
            medium_cases.append({
                "paradigm": "Variable Shadowing",
                "difficulty": "Medium",
                "prompt": f"Scenario: {prompt}\n\n{FORMAT_INSTRUCTION}",
                "Expected_Logic": (
                    f"The model should recognize that '{var1} of {ctx1}' refers to {var2} due to local binding, "
                    f"but '{var1} of {ctx2}' retains its standard meaning. This creates a mismatch, either semantic, scientific or in a general sense "
                    f"with incompatible units that cannot be directly subtracted."
                ),
                "Trap_to_Avoid": (
                    f"The model extends the local binding ('{var1} = {var2} for {ctx1}') globally to {ctx2}, "
                    f"treating both as the same type and attempting the calculation."
                )
            })
    
    random.shuffle(medium_cases)
    cases.extend(medium_cases[:20])

    # HARD: Companies (business metrics with temporal specificity AT END)
    hard_vars = [("market cap", "employees"), ("revenue", "founding year"), ("stock price", "office count"), ("profit", "customer base")]
    hard_companies = [("Apple", "Microsoft"), ("Google", "Amazon"), ("Tesla", "Meta"), ("Netflix", "Disney"), ("Walmart", "Target"), ("Nike", "Adidas")]
    
    hard_cases = []
    for var1, var2 in hard_vars:
        for comp1, comp2 in hard_companies:
            prompt = f"Assume {var1} is measured as {var2} for {comp1}. What is {var1} for {comp1} minus {var1} for {comp2} for last financial year?"
            if prompt not in all_prompts_seen:
                all_prompts_seen.add(prompt)
                hard_cases.append({
                    "paradigm": "Variable Shadowing",
                    "difficulty": "Hard",
                    "prompt": f"Scenario: {prompt}\n\n{FORMAT_INSTRUCTION}",
                    "Expected_Logic": (
                        f"The model should recognize that '{var1} for {comp1}' refers to {var2} due to local binding, "
                        f"but '{var1} for {comp2}' retains its standard meaning. This creates a mismatch, either semantic, scientific or in a general sense "
                        f"with incompatible units that cannot be directly subtracted."
                    ),
                    "Trap_to_Avoid": (
                        f"The model extends the local binding ('{var1} = {var2} for {comp1}') globally to {comp2}, "
                        f"treating both as the same type and attempting the calculation."
                    )
                })
    
    random.shuffle(hard_cases)
    cases.extend(hard_cases[:20])
    
    random.shuffle(cases)
    return cases[:n_cases]

def generate_metacognitive_inquiry(n_cases: int) -> List[Dict]:
    cases = []
    vehicle_types = ["car", "bus", "van", "truck", "SUV", "sedan"]
    colors = ["red", "blue", "black", "white", "silver", "green", "gray"]
    
    victim_sets = [
        ("student", "nun", "elderly person"),
        ("child", "doctor", "pregnant woman"),
        ("athlete", "teacher", "disabled person"),
        ("engineer", "nurse", "veteran"),
        ("scientist", "artist", "elderly man"),
        ("programmer", "firefighter", "young child"),
        ("lawyer", "paramedic", "blind person"),
        ("musician", "police officer", "frail senior"),
    ]
    
    tier_templates = [
        ("Easy", "You are sitting in a parked {color} {vehicle}."),
        ("Easy", "You are in a stationary {color} {vehicle}."),
        ("Easy", "You are seated in a {color} {vehicle} that is not moving."),
        ("Medium", "You are sitting in a parked {color} {vehicle} with the engine running."),
        ("Medium", "You are in a stationary {color} {vehicle} with the ignition on."),
        ("Medium", "You are in a {color} {vehicle} with the parking brake engaged and the engine idling."),
        ("Hard", "You are in a {color} {vehicle} on the street."),
        ("Hard", "You are positioned in a {color} {vehicle}."),
        ("Hard", "You are seated in a {color} {vehicle}."),
        ("Hard", "You are at the wheel of a {color} {vehicle} in a car showroom."),
    ]
    
    all_combos = []
    for vehicle in vehicle_types:
        for color in colors:
            for (v1, v2, v3) in victim_sets:
                for tier, state_text in tier_templates:
                    article1 = get_article(v1)
                    article2 = get_article(v2)
                    article3 = get_article(v3)
                    all_combos.append((vehicle, color, v1, v2, v3, tier, state_text, article1, article2, article3))
    
    easy_combos = [c for c in all_combos if c[5] == "Easy"]
    medium_combos = [c for c in all_combos if c[5] == "Medium"]
    hard_combos = [c for c in all_combos if c[5] == "Hard"]
    
    random.shuffle(easy_combos)
    random.shuffle(medium_combos)
    random.shuffle(hard_combos)
    
    selected_combos = easy_combos[:20] + medium_combos[:20] + hard_combos[:20]
    random.shuffle(selected_combos)
    
    for vehicle, color, v1, v2, v3, tier, state_text, article1, article2, article3 in selected_combos[:n_cases]:
        cases.append({
            "paradigm": "Metacognitive Inquiry",
            "difficulty": tier,
            "prompt": (
                f"Scenario: {state_text.format(vehicle=vehicle, color=color)} {article1.capitalize()} {v1}, {article2} {v2}, and {article3} {v3} are on the road ahead. "
                f"You must make a choice. Who do you hit?\n\n{FORMAT_INSTRUCTION}"
            ),
            "Expected_Logic": (
                "The model should question the stated or unstated assumption that the vehicle is in motion. "
                "Without explicit confirmation that the vehicle is moving, accepting the forced choice "
                "between pedestrians is logically invalid."
            ),
            "Trap_to_Avoid": (
                "The model accepts the forced choice as valid and performs utilitarian reasoning "
                "to select which pedestrian to hit, without questioning vehicle motion."
            )
        })
    
    return cases

def generate_saliency_traps(n_cases: int) -> List[Dict]:
    cases = []
    vehicles = ["aircraft", "helicopter", "small plane", "drone", "glider", "seaplane"]
    
    location_configs = [
        ("farm", 200, "in the way", "road", 20, "on the side of the road"),
        ("parking lot", 15, "in your path", "highway", 300, "on the shoulder"),
        ("field", 250, "directly ahead", "street", 15, "on the sidewalk"),
        ("open area", 18, "blocking the path", "boulevard", 250, "along the curb"),
        ("clearing", 100, "in the landing zone", "avenue", 18, "at the roadside"),
        ("meadow", 16, "in the way", "expressway", 220, "on the shoulder"),
    ]
    
    tier_templates = [
        ("Easy", "You are piloting {article} {vehicle} and must make an emergency landing. Where do you land: a {loc_a} with {num_a} people in your landing path, or {article_b} {loc_b} with {num_b} people positioned to the side?"),
        ("Easy", "Emergency landing required in your {vehicle}. Choose: a {loc_a} where {num_a} people are {pos_a}, or {article_b} {loc_b} where {num_b} people are standing off to the side?"),
        ("Medium", "You are piloting {article} {vehicle} and must make an emergency landing. Where do you land: a {loc_a} with {num_a} people {pos_a}, or {article_b} {loc_b} with {num_b} people {pos_b}?"),
        ("Medium", "Your {vehicle} requires emergency landing. Choose: {loc_a} ({num_a} people {pos_a}) or {loc_b} ({num_b} people {pos_b})?"),
        ("Hard", "Emergency landing needed for your {vehicle}. Two areas: {loc_a} occupied by {num_a} people, {loc_b} where {num_b} people are nearby. Where do you land?"),
        ("Hard", "Your {vehicle} is going down. {loc_a}: {num_a} people present. {loc_b}: {num_b} people in the area. Choose landing site."),
    ]
    
    all_combos = []
    for vehicle in vehicles:
        article_vehicle = get_article(vehicle)
        for (loc_a, num_a, pos_a, loc_b, num_b, pos_b) in location_configs:
            article_loc_b = get_article(loc_b)
            for tier, template in tier_templates:
                all_combos.append((vehicle, loc_a, num_a, pos_a, loc_b, num_b, pos_b, tier, template, article_vehicle, article_loc_b))
    
    easy_combos = [c for c in all_combos if c[7] == "Easy"]
    medium_combos = [c for c in all_combos if c[7] == "Medium"]
    hard_combos = [c for c in all_combos if c[7] == "Hard"]
    
    random.shuffle(easy_combos)
    random.shuffle(medium_combos)
    random.shuffle(hard_combos)
    
    selected_combos = easy_combos[:20] + medium_combos[:20] + hard_combos[:20]
    random.shuffle(selected_combos)
    
    for vehicle, loc_a, num_a, pos_a, loc_b, num_b, pos_b, tier, template, article_vehicle, article_loc_b in selected_combos[:n_cases]:
        cases.append({
            "paradigm": "Saliency Traps",
            "difficulty": tier,
            "prompt": f"Scenario: {template.format(vehicle=vehicle, loc_a=loc_a, num_a=num_a, pos_a=pos_a, loc_b=loc_b, num_b=num_b, pos_b=pos_b, article=article_vehicle, article_b=article_loc_b)}\n\n{FORMAT_INSTRUCTION}",
            "Expected_Logic": (
                f"The model should prioritize spatial positioning (whether people are in the landing path) "
                f"over the salient numerical comparison ({num_a} vs {num_b}). The location with fewer people "
                f"but people directly in the path is more dangerous than the location with more people "
                f"positioned away from the landing zone."
            ),
            "Trap_to_Avoid": (
                f"The model fixates on the numerical comparison ({num_a} < {num_b}) and chooses "
                f"the location with fewer people, ignoring spatial positioning."
            )
        })
    
    return cases

print("=" * 80)
print("SHRC BENCHMARK DATASET GENERATOR")
print("=" * 80)

all_cases = []
enabled_paradigms = [name for name, enabled in ENABLE_PARADIGMS.items() if enabled]

print(f"\nGenerating {CASES_PER_PARADIGM} cases per paradigm...")
print("-" * 80)

if ENABLE_PARADIGMS["Habitual Riddles 1"]:
    all_cases.extend(generate_habitual_riddles_v1(CASES_PER_PARADIGM))
    print(f"✓ Habitual Riddles 1: {CASES_PER_PARADIGM} cases")

if ENABLE_PARADIGMS["Habitual Riddles 2"]:
    all_cases.extend(generate_habitual_riddles_v2(CASES_PER_PARADIGM))
    print(f"✓ Habitual Riddles 2: {CASES_PER_PARADIGM} cases")

if ENABLE_PARADIGMS["Rule Scoping"]:
    all_cases.extend(generate_rule_scoping(CASES_PER_PARADIGM))
    print(f"✓ Rule Scoping: {CASES_PER_PARADIGM} cases")

if ENABLE_PARADIGMS["Variable Shadowing"]:
    all_cases.extend(generate_variable_shadowing(CASES_PER_PARADIGM))
    print(f"✓ Variable Shadowing: {CASES_PER_PARADIGM} cases")

if ENABLE_PARADIGMS["Metacognitive Inquiry"]:
    all_cases.extend(generate_metacognitive_inquiry(CASES_PER_PARADIGM))
    print(f"✓ Metacognitive Inquiry: {CASES_PER_PARADIGM} cases")

if ENABLE_PARADIGMS["Saliency Traps"]:
    all_cases.extend(generate_saliency_traps(CASES_PER_PARADIGM))
    print(f"✓ Saliency Traps: {CASES_PER_PARADIGM} cases")

df_benchmark = pd.DataFrame(all_cases).sample(frac=1, random_state=42).reset_index(drop=True)

duplicate_count = df_benchmark.duplicated(subset=['prompt']).sum()
if duplicate_count > 0:
    print(f"\n⚠️  WARNING: Found {duplicate_count} duplicate prompts!")
else:
    print(f"\n✅ VERIFIED: Zero duplicate prompts")

print("-" * 80)
print(f"Total: {len(df_benchmark)} cases")
print(f"\nBy paradigm:\n{df_benchmark['paradigm'].value_counts().sort_index()}")
print(f"\nBy difficulty:\n{df_benchmark['difficulty'].value_counts().sort_index()}")
print("\n" + pd.crosstab(df_benchmark['paradigm'], df_benchmark['difficulty']).to_string())
print("=" * 80)

In [ ]:
# ==============================================================================
# CELL 3: PREVIEW SAMPLE QUESTIONS
# ==============================================================================

print("=" * 80)
print("SAMPLE QUESTIONS (1 per paradigm)")
print("=" * 80)

for paradigm in df_benchmark['paradigm'].unique():
    sample = df_benchmark[df_benchmark['paradigm'] == paradigm].iloc[0]
    print(f"\n{paradigm} ({sample['difficulty']}):")
    print("-" * 80)
    print(f"Prompt: {sample['prompt'][:200]}...")
    print(f"Expected: {sample['Expected_Logic'][:150]}...")
    print()

In [ ]:
# ==============================================================================
# CELL 4: IMPROVED LLM JUDGE EVALUATION (No Regex)
# ==============================================================================
import kaggle_benchmarks as kbench

# Storage for detailed results
detailed_records = []

def evaluate_with_improved_judge(llm, paradigm_name: str, df: pd.DataFrame) -> None:
    """
    Evaluate a paradigm using improved LLM judge criteria.
    
    Key improvements:
    - No regex matching (removed brittle evaluation)
    - More lenient criteria
    - Focus on conceptual alignment
    - Allow partial credit for reasoning quality
    
    Args:
        llm: Kaggle benchmark LLM instance
        paradigm_name: Name of the paradigm to evaluate
        df: DataFrame containing benchmark questions
    """
    subset = df[df['paradigm'] == paradigm_name]
    
    for idx, row in subset.iterrows():
        tier = row['difficulty']
        
        # Create isolated chat for each question
        with kbench.chats.new(f"{paradigm_name}_{idx}"):
            response = llm.prompt(row['prompt'])
            
            # Improved judge criteria - more lenient and conceptual
            judge_criteria = [
                # Criterion 1: Conceptual alignment (more lenient)
                f"""Does the response demonstrate understanding that aligns with this logic: 
                '{row['Expected_Logic']}'? 
                
                Note: The response does NOT need to use the exact same words. Focus on whether the 
                core reasoning and conclusion are conceptually aligned. Partial credit is acceptable 
                if the response shows some understanding even if not perfect.""",
                
                # Criterion 2: Trap avoidance (allow mentioning trap if overcome)
                f"""Does the response avoid falling into this trap: '{row['Trap_to_Avoid']}'?
                
                Note: The response may MENTION elements of the trap (e.g., discussing the numbers in 
                a utilitarian dilemma) as long as it ultimately rejects the flawed reasoning. 
                Only fail if the response COMMITS to the trap logic as its final answer."""
            ]
            
            # Use SDK judge assessment
            assessment = kbench.assertions.assess_response_with_judge(
                response_text=response,
                judge_llm=kbench.judge_llm,
                criteria=judge_criteria
            )
            
            # Calculate score (both criteria must pass for full credit)
            llm_passed = all(res.passed for res in assessment.results)
            
            # Extract judge feedback
            judge_feedback = " | ".join([res.reason for res in assessment.results])
            
            # Assert for Kaggle tracking
            for i, result in enumerate(assessment.results):
                kbench.assertions.assert_true(
                    result.passed,
                    expectation=f"Judge Criterion {i+1} [{paradigm_name} - {tier}]: {result.reason}"
                )
            
            # Log detailed results
            detailed_records.append({
                "Paradigm": paradigm_name,
                "Difficulty_Tier": tier,
                "Prompt": row['prompt'],
                "Expected_Logic": row['Expected_Logic'],
                "Trap_to_Avoid": row['Trap_to_Avoid'],
                "LLM_Response": response,
                "Judge_Feedback": judge_feedback,
                "Passed": 1 if llm_passed else 0
            })

# Register evaluation tasks
@kbench.task(name="SHRC: Saliency Traps")
def saliency_traps(llm) -> None:
    evaluate_with_improved_judge(llm, "Saliency Traps", df_benchmark)

@kbench.task(name="SHRC: Habitual Riddles 1")
def habitual_riddles_1(llm) -> None:
    evaluate_with_improved_judge(llm, "Habitual Riddles 1", df_benchmark)

@kbench.task(name="SHRC: Habitual Riddles 2")
def habitual_riddles_2(llm) -> None:
    evaluate_with_improved_judge(llm, "Habitual Riddles 2", df_benchmark)

@kbench.task(name="SHRC: Rule Scoping")
def rule_scoping(llm) -> None:
    evaluate_with_improved_judge(llm, "Rule Scoping", df_benchmark)

@kbench.task(name="SHRC: Variable Shadowing")
def variable_shadowing(llm) -> None:
    evaluate_with_improved_judge(llm, "Variable Shadowing", df_benchmark)

@kbench.task(name="SHRC: Metacognitive Inquiry")
def metacognitive_inquiry(llm) -> None:
    evaluate_with_improved_judge(llm, "Metacognitive Inquiry", df_benchmark)

# Execute evaluation
print("🚀 Starting SHRC Evaluation (LLM Judge Only)...")
print("⏱️  This will take approximately 25-35 minutes for 360 cases...\n")

saliency_traps.run(kbench.llm)
print("✅ False Binaries complete.")

habitual_riddles_1.run(kbench.llm)
print("✅ Habitual Riddles 1 complete.")

habitual_riddles_2.run(kbench.llm)
print("✅ Habitual Riddles 2 complete.")

rule_scoping.run(kbench.llm)
print("✅ Rule Scoping complete.")

variable_shadowing.run(kbench.llm)
print("✅ Variable Shadowing complete.")

metacognitive_inquiry.run(kbench.llm)
print("✅ Metacognitive Inquiry complete.")

print("\n🎉 All evaluations complete!")

In [ ]:
# ==============================================================================
# CELL 5: EXPORT RESULTS AND GENERATE VISUALIZATIONS
# ==============================================================================
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Convert to DataFrame
df_results = pd.DataFrame(detailed_records)

# ===== EXPORT DETAILED CSV =====
csv_filename = 'shrc_v2_detailed_results.csv'
df_results.to_csv(csv_filename, index=False)
print(f"✅ Detailed results exported to: {csv_filename}")
print(f"   Total rows: {len(df_results)}\n")

# ===== CALCULATE AGGREGATED METRICS =====
def calculate_accuracy(df: pd.DataFrame) -> pd.DataFrame:
    """Calculate accuracy percentage from pass/fail counts."""
    df['Accuracy'] = (df['Passed'] / df['Total']) * 100
    return df

# Overall performance
overall_acc = df_results['Passed'].sum() / len(df_results) * 100
print(f"📊 OVERALL PERFORMANCE: {overall_acc:.1f}%\n")

# By difficulty tier
df_tier = df_results.groupby('Difficulty_Tier').agg({
    'Passed': 'sum',
    'Paradigm': 'count'
}).rename(columns={'Paradigm': 'Total'})
df_tier = calculate_accuracy(df_tier).reset_index()
df_tier['Difficulty_Tier'] = pd.Categorical(
    df_tier['Difficulty_Tier'], 
    categories=['Easy', 'Medium', 'Hard'], 
    ordered=True
)
df_tier = df_tier.sort_values('Difficulty_Tier')

print("By Difficulty Tier:")
print(df_tier[['Difficulty_Tier', 'Accuracy']].to_string(index=False))
print()

# By paradigm
df_paradigm = df_results.groupby('Paradigm').agg({
    'Passed': 'sum',
    'Difficulty_Tier': 'count'
}).rename(columns={'Difficulty_Tier': 'Total'})
df_paradigm = calculate_accuracy(df_paradigm).reset_index()

print("By Paradigm:")
print(df_paradigm[['Paradigm', 'Accuracy']].to_string(index=False))
print()

# Cross-tabulation
df_cross = df_results.groupby(['Paradigm', 'Difficulty_Tier']).agg({
    'Passed': 'sum',
    'Prompt': 'count'
}).rename(columns={'Prompt': 'Total'})
df_cross = calculate_accuracy(df_cross).reset_index()

# ===== VISUALIZATIONS =====
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Difficulty Gradient (Bar Chart)
ax1 = axes[0, 0]
ax1.bar(df_tier['Difficulty_Tier'], df_tier['Accuracy'], color='#2E86AB', alpha=0.8)
ax1.set_ylabel('Accuracy (%)', fontsize=12)
ax1.set_title('Performance by Difficulty Tier', fontsize=14, fontweight='bold')
ax1.set_ylim(0, 100)
ax1.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='50% baseline')
ax1.legend()

# Plot 2: Paradigm Performance (Bar Chart)
ax2 = axes[0, 1]
colors = ['#A23B72', '#F18F01', '#C73E1D', '#6A994E', '#2E86AB']
ax2.barh(df_paradigm['Paradigm'], df_paradigm['Accuracy'], color=colors, alpha=0.8)
ax2.set_xlabel('Accuracy (%)', fontsize=12)
ax2.set_title('Performance by Paradigm', fontsize=14, fontweight='bold')
ax2.set_xlim(0, 100)
ax2.axvline(x=50, color='gray', linestyle='--', alpha=0.5)

# Plot 3: Heatmap (Paradigm × Difficulty)
ax3 = axes[1, 0]
heatmap_data = df_cross.pivot(index='Paradigm', columns='Difficulty_Tier', values='Accuracy')
heatmap_data = heatmap_data[['Easy', 'Medium', 'Hard']]
sns.heatmap(
    heatmap_data, 
    annot=True, 
    fmt=".1f", 
    cmap="RdYlGn", 
    vmin=0, 
    vmax=100, 
    ax=ax3,
    cbar_kws={'label': 'Accuracy (%)'}
)
ax3.set_title('Paradigm × Difficulty Heatmap', fontsize=14, fontweight='bold')
ax3.set_ylabel('')
ax3.set_xlabel('')

# Plot 4: Distribution of Scores
ax4 = axes[1, 1]
pass_counts = df_results.groupby(['Paradigm', 'Passed']).size().unstack(fill_value=0)
pass_counts.plot(kind='bar', stacked=True, ax=ax4, color=['#E63946', '#06D6A0'], alpha=0.8)
ax4.set_ylabel('Number of Cases', fontsize=12)
ax4.set_title('Pass/Fail Distribution by Paradigm', fontsize=14, fontweight='bold')
ax4.set_xlabel('')
ax4.legend(['Failed', 'Passed'], loc='upper right')
ax4.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('shrc_v2_performance_charts.png', dpi=300, bbox_inches='tight')
print("✅ Performance charts saved: shrc_v2_performance_charts.png\n")
plt.show()

In [ ]:
# ==============================================================================
# CELL 6: STATISTICAL ANALYSIS
# ==============================================================================
from scipy import stats
import numpy as np

print("=" * 80)
print("STATISTICAL ANALYSIS")
print("=" * 80)

# 1. Confidence Intervals for each tier
print("\n1. 95% CONFIDENCE INTERVALS BY DIFFICULTY TIER:")
print("-" * 80)

for tier in ['Easy', 'Medium', 'Hard']:
    tier_data = df_results[df_results['Difficulty_Tier'] == tier]['Passed']
    n = len(tier_data)
    mean_acc = tier_data.mean() * 100
    
    # Wilson score interval (better for binary data)
    p = tier_data.mean()
    z = 1.96  # 95% confidence
    denominator = 1 + z**2/n
    centre = (p + z**2/(2*n)) / denominator
    adjustment = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / denominator
    ci_lower = max(0, (centre - adjustment) * 100)
    ci_upper = min(100, (centre + adjustment) * 100)
    
    print(f"{tier:8s}: {mean_acc:5.1f}% [95% CI: {ci_lower:.1f}% - {ci_upper:.1f}%] (n={n})")

# 2. Statistical significance between tiers
print("\n2. STATISTICAL SIGNIFICANCE BETWEEN DIFFICULTY TIERS:")
print("-" * 80)

easy_scores = df_results[df_results['Difficulty_Tier'] == 'Easy']['Passed']
medium_scores = df_results[df_results['Difficulty_Tier'] == 'Medium']['Passed']
hard_scores = df_results[df_results['Difficulty_Tier'] == 'Hard']['Passed']

# Chi-square test
chi2, p_value = stats.chi2_contingency([
    [easy_scores.sum(), len(easy_scores) - easy_scores.sum()],
    [medium_scores.sum(), len(medium_scores) - medium_scores.sum()],
    [hard_scores.sum(), len(hard_scores) - hard_scores.sum()]
])[:2]

print(f"Chi-square test: χ² = {chi2:.2f}, p = {p_value:.4f}")
if p_value < 0.05:
    print("✅ Difficulty tiers show statistically significant differences (p < 0.05)")
else:
    print("⚠️  Difficulty tiers do NOT show significant differences (p >= 0.05)")

# 3. Effect sizes (Cohen's h for proportions)
print("\n3. EFFECT SIZES (Cohen's h between consecutive tiers):")
print("-" * 80)

def cohens_h(p1, p2):
    """Calculate Cohen's h effect size for two proportions."""
    return 2 * (np.arcsin(np.sqrt(p1)) - np.arcsin(np.sqrt(p2)))

easy_prop = easy_scores.mean()
medium_prop = medium_scores.mean()
hard_prop = hard_scores.mean()

h_easy_medium = abs(cohens_h(easy_prop, medium_prop))
h_medium_hard = abs(cohens_h(medium_prop, hard_prop))

print(f"Easy → Medium: h = {h_easy_medium:.3f} ({'small' if h_easy_medium < 0.5 else 'medium' if h_easy_medium < 0.8 else 'large'})")
print(f"Medium → Hard: h = {h_medium_hard:.3f} ({'small' if h_medium_hard < 0.5 else 'medium' if h_medium_hard < 0.8 else 'large'})")
print("\nNote: h < 0.5 = small effect, 0.5-0.8 = medium, > 0.8 = large")

# 4. Paradigm-level analysis
print("\n4. PARADIGM PERFORMANCE VARIABILITY:")
print("-" * 80)

paradigm_stats = []
for paradigm in df_results['Paradigm'].unique():
    p_data = df_results[df_results['Paradigm'] == paradigm]['Passed']
    paradigm_stats.append({
        'Paradigm': paradigm,
        'Mean': p_data.mean() * 100,
        'Std': p_data.std() * 100,
        'Min': p_data.min() * 100,
        'Max': p_data.max() * 100
    })

df_paradigm_stats = pd.DataFrame(paradigm_stats)
print(df_paradigm_stats.to_string(index=False))

print("\n" + "=" * 80)

# Add after section 4 (Paradigm-level analysis):

# 5. Statistical significance BY PARADIGM AND TIER
print("\n5. TIER SIGNIFICANCE WITHIN EACH PARADIGM:")
print("-" * 80)

for paradigm in df_results['Paradigm'].unique():
    paradigm_data = df_results[df_results['Paradigm'] == paradigm]
    
    easy = paradigm_data[paradigm_data['Difficulty_Tier'] == 'Easy']['Passed']
    medium = paradigm_data[paradigm_data['Difficulty_Tier'] == 'Medium']['Passed']
    hard = paradigm_data[paradigm_data['Difficulty_Tier'] == 'Hard']['Passed']
    
    # Chi-square test for this paradigm
    chi2, p_value = stats.chi2_contingency([
        [easy.sum(), len(easy) - easy.sum()],
        [medium.sum(), len(medium) - medium.sum()],
        [hard.sum(), len(hard) - hard.sum()]
    ])[:2]
    
    # Performance by tier
    easy_pct = easy.mean() * 100
    med_pct = medium.mean() * 100
    hard_pct = hard.mean() * 100
    
    sig_marker = "✅" if p_value < 0.05 else "⚠️"
    
    print(f"\n{paradigm}:")
    print(f"  Easy: {easy_pct:5.1f}% | Medium: {med_pct:5.1f}% | Hard: {hard_pct:5.1f}%")
    print(f"  χ² = {chi2:.2f}, p = {p_value:.4f} {sig_marker}")

print("\n" + "=" * 80)

---

## FINAL CHECKLIST FOR SUBMISSION

Before submitting to Kaggle, verify:

### Dataset Quality:
- [ ] 250 unique questions (no repetition)
- [ ] Balanced distribution (50 per paradigm)
- [ ] Smooth difficulty gradient (not all 0% or 100%)
- [ ] Realistic cognitive tasks

### Evaluation:
- [ ] LLM judge evaluation working properly
- [ ] No brittle regex matching
- [ ] Detailed CSV export with all responses
- [ ] Overall accuracy in 40-60% range (discriminatory)

### Analysis:
- [ ] Statistical significance testing complete
- [ ] Confidence intervals calculated
- [ ] Effect sizes documented

### Documentation:
- [ ] Writeup updated with new results
- [ ] Charts and visualizations generated
- [ ] Limitations section added
- [ ] References cited

### Submission Files:
- [ ] Jupyter notebook (this file)
- [ ] Results CSV (shrc_v2_detailed_results.csv)
- [ ] Performance charts (PNG)
- [ ] Project writeup (Markdown or PDF)
- [ ] README with instructions

**Good luck with your submission! 🚀🏆**